# Feature Extraction: Skills, Experience, and Requirements
**CSC 366: Principles of Data Science | Group 02**

Reads `jobs_combined.csv` and adds new columns pulled from each job `description`. Saves the result as `jobs_features.csv` (the input file is not changed).

**Upload `jobs_combined.csv` before running.**

## Load the data

In [8]:
import pandas as pd
import numpy as np
import re

dfJobs = pd.read_csv("jobs_combined.csv")
print(dfJobs.shape)

(9437, 27)


## Skill list

The notebook tracks 38 skills. A skill was included if it showed up in at least 5% of job postings in either dataset. Two AI tools, MCP and LangChain/RAG, were also included even though they are below 5%, because they are growing quickly in the Hacker News posts.

Each skill has a search pattern used to find it in the job descriptions. The patterns only match whole words, so searching for "Java" does not also count "JavaScript."

The C language has its own rule, because the letter "C" also shows up in phrases like "Series C" and "D.C." It only counts when it appears next to other languages or programming words, such as "C/C++" or "C, Java."

**AI assistance:** The job description data was given to AI (Claude) ahead of time, so it could count how often each skill actually appeared and write search patterns that match the way skills are written in these postings. The group made the final decisions, including the 5% cutoff and keeping C.

In [9]:
# Each skill and the pattern used to find it in the job description.
# \b = whole word only, so "java" does not match "javascript".
skillPatterns = {
    # Languages
    "Python": r"\bpython\b",
    "Java": r"\bjava\b(?!script)",
    "JavaScript": r"\bjavascript\b",
    "TypeScript": r"\btypescript\b",
    "C++": r"c\+\+",
    "C#/.NET": r"c#|(?<![\w/.])\.net\b|dotnet",
    "Go": r"\bgolang\b|\bgo\b(?=\s*(?:,|/|\)|and\b|or\b|lang))",
    "Rust": r"\brust\b",
    "Ruby": r"\bruby\b|\brails\b",
    "SQL": r"\bsql\b|mysql",
    "Postgres": r"postgres",
    "NoSQL": r"nosql|mongodb",
    # Web
    "React": r"\breact\b",
    "Angular": r"\bangular\b",
    "Node.js": r"node\.?js|\bnode\b",
    "Next.js": r"next\.?js",
    "HTML/CSS": r"\bhtml\b|\bcss\b",
    "Django/Flask/FastAPI": r"django|flask|fastapi",
    "Spring": r"spring boot|spring framework|\bspring\b(?!\s*(?:\d|semester|break|term|internship|intern|cohort|season))",
    "APIs": r"\brest\b|restful|\bapis?\b",
    # Cloud / DevOps
    "AWS": r"\baws\b|amazon web services",
    "Azure": r"\bazure\b",
    "GCP": r"\bgcp\b|google cloud",
    "Docker": r"\bdocker\b",
    "Kubernetes": r"kubernetes|\bk8s\b",
    "Terraform": r"terraform",
    "CI/CD": r"ci/cd|continuous integration",
    "Linux": r"\blinux\b|\bunix\b",
    "Git": r"\bgit\b|github",
    # Data / AI
    "Machine learning": r"machine learning|\bml\b",
    "PyTorch/TensorFlow": r"pytorch|tensorflow|deep learning",
    "AI/LLMs": r"\bllms?\b|generative ai|\bgenai\b",
    "AI agents": r"agentic|ai agents?|llm agents?|autonomous agents?|coding agents?|multi-agent",
    "LangChain/RAG": r"langchain|\brag\b",
    "MCP": r"\bmcp\b|model context protocol",
    # Other
    "Embedded/RTOS": r"embedded (?:systems?|software|c\b|development|linux|devices?|engineer)|firmware|\brtos\b",
    "Agile": r"\bagile\b|scrum",
}

# C is checked separately and case-sensitive: a capital C next to other
# languages or programming words ("C/C++", "C, Java", "in C"), but not
# "Series C", "Objective-C", "D.C.", or "U.S.C."
cPattern = (r"(?<!Series )(?<!Objective-)(?<![\w.\-])C(?![\w.+#\-])"
            r"(?=\s*(?:/|,|&|\)|\band\b|\bor\b|programming|language|code|developer))"
            r"|(?:\bin|\(|,)\s*C(?![\w.+#\-])")

print(f"Skills tracked: {len(skillPatterns) + 1}")

Skills tracked: 38


## Find the skills in each posting

`findSkills` checks the description against every pattern and returns the matches as one string.

**AI assistance** AI was used to create the test



In [10]:
def findSkills(description):
    found = []
    text = description.lower()

    for skill, pattern in skillPatterns.items():
        if re.search(pattern, text):
            found.append(skill)

    # C uses the original text (capital C)
    if re.search(cPattern, description):
        found.append("C")

    return ", ".join(found)

# Test
print(findSkills("Experience with Python, C/C++, and AWS. Series C startup."))   # Python, C++, AWS, C
print(findSkills("We use JavaScript and React"))                               # JavaScript, React

dfJobs["skills"] = dfJobs["description"].apply(findSkills)
dfJobs[["title", "skills"]].head(10)

Python, C++, AWS, C
JavaScript, React


,title,skills
0,Software Engineer (Entry Level),"Python, Java, JavaScript, C++, Spring, APIs, D..."
1,"Software Engineer I, Entry-Level Graduation Da...","Python, Java, SQL, APIs, CI/CD"
2,Entry Level Software Engineer,"Python, Java, C++, Linux, Git, Embedded/RTOS, C"
3,Campus Graduate Masters Full-Time Engineer - 2...,"Python, Java, JavaScript, TypeScript, C#/.NET,..."
4,Entry-Level Quantitative Software Engineer I,
5,Software Engineer: Credit Card/Fraud,
6,software engineer (entry level),"Python, Java, C++, Spring"
7,Associate Software Engineer (Entry Level)/Data...,"Python, Java, Spring, APIs, Docker, Kubernetes..."
8,Entry Level Software Engineer Jobs,"Python, Java, C++, Linux, Git, Embedded/RTOS, C"
9,Remote Software Engineer (Java/Angular) - Entr...,"Java, Angular, CI/CD, Agile"


## True/False column for each skill

Each posting now has a `skills` column listing its skills, like "Python, SQL, AWS." This is easy to read, but hard to count with, because every cell holds several skills together.

To make counting easier, a loop creates one True/False column for each skill, such as `has_python` or `has_sql`. A posting gets `True` if it mentions that skill. With these columns, simple tools like `.sum()` and `.groupby()` can answer questions like "how many postings mention Python?" or "do postings with Python pay more?"

`skill_count` is the number of skills found in each posting.

**AI assistance:** The group first planned to store the skills only as a list in one column. AI (Claude) recommended also adding a True/False column for each skill, because it makes counting and comparing much simpler. Both were kept: the `skills` list for reading, and the `has_` columns for analysis.

In [11]:
# Reuse the skill/pattern dictionary from before
allSkills = list(skillPatterns.keys()) + ["C"]

# Creates column names
def makeColumnName(skill):
    # "C++" -> "has_cpp", "C#/.NET" -> "has_csharp_net", "AI agents" -> "has_ai_agents"
    name = skill.lower().replace("++", "pp").replace("#", "sharp")
    name = name.replace("/", "_").replace(".", "").replace(" ", "_")
    return "has_" + name

skillLists = dfJobs["skills"].str.split(", ")

# AI assistance: AI (Claude) wrote this loop. The group reviewed it, ran it,
# and checked that each has_ column matched the skills list.

for skill in allSkills:                       # outer loop: each of the 38 skills
    hasSkill = []                             # empty list for this skill's column
    for skillList in skillLists:              # inner loop: each posting
        hasSkill.append(skill in skillList)   # True if the skill is in that posting's list
    dfJobs[makeColumnName(skill)] = hasSkill  # save the list as a new column

# Number of skills per posting
dfJobs["skill_count"] = dfJobs["skills"].str.split(", ").str.len()
dfJobs.loc[dfJobs["skills"] == "", "skill_count"] = 0

print([c for c in dfJobs.columns if c.startswith("has_")])

['has_python', 'has_java', 'has_javascript', 'has_typescript', 'has_cpp', 'has_csharp_net', 'has_go', 'has_rust', 'has_ruby', 'has_sql', 'has_postgres', 'has_nosql', 'has_react', 'has_angular', 'has_nodejs', 'has_nextjs', 'has_html_css', 'has_django_flask_fastapi', 'has_spring', 'has_apis', 'has_aws', 'has_azure', 'has_gcp', 'has_docker', 'has_kubernetes', 'has_terraform', 'has_ci_cd', 'has_linux', 'has_git', 'has_machine_learning', 'has_pytorch_tensorflow', 'has_ai_llms', 'has_ai_agents', 'has_langchain_rag', 'has_mcp', 'has_embedded_rtos', 'has_agile', 'has_c']


## Years of experience

Processing years of experience from text. Each posting asked experience like: "2+ years of experience" or "3-5 years of professional experience."

**What counts:**
- A number, then "years," then the word "experience" a few words later.
- A number, then "years," then a work word right after, such as "3+ years professional iOS development."
- Numbers written as words, like "two years," are converted to numbers.

**How the number is chosen:**
- For a range like "3-5 years," the **lowest** number is used (3), since that is the minimum needed to apply.
- If a posting mentions more than one requirement, the **highest** one is used, since that is the strictest.

**What is skipped:**
- Maximums, like "up to 2 years" or "less than 2 years."
- Degree substitutions, like "4 years of experience may substitute for a degree."
- Numbers over 15, which are usually about the company ("for over 65 years").

If a posting doesn't mention years of experience, the value is left missing, not set to 0.

**AI assistance:** The job description data was given to AI (Claude) ahead of time. Claude looked at how experience requirements were actually written in the postings, including samples of what the pattern matched and missed, and wrote the regex pattern and the rules for choosing the number. The skip rules for maximums and degree substitutions were added after a second review (from ChatGPT) pointed out those cases, and Claude updated the pattern to handle them.

In [12]:
# Number words that sometimes appear instead of digits
numberWords = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
               "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10}

# A number (or range), then "years", then either:
#   - "experience" within a few words   ("2+ years of professional experience")
#   - a work word right after "years"   ("3+ years professional iOS development",
#                                        "2 years in software development")
yearsPattern = (r"(\d{1,2}|one|two|three|four|five|six|seven|eight|nine|ten)\s*\+?\s*"
                r"(?:(?:-|–|to)\s*\d{1,2}\s*\+?\s*)?"
                r"(?:years?|yrs?)\b[’']?\s*"
                r"(?:(?:of\s+)?(?:[\w\-‑/+#.,]+\s+){0,4}?experience"
                r"|(?:of\s+|in\s+|with\s+)?(?:professional|industry|hands-on|relevant|work|software|programming|coding|development|engineering)\b)")

# Phrases that mean the number is NOT a minimum requirement:
#   degree substitutions ("4 years of experience may substitute for a degree")
#   maximums ("up to 2 years", "no more than 2 years", "less than 2 years")
substituteWords = r"in lieu|substitut|in place of|instead of a degree"
maximumWords = r"(?:up to|no more than|not more than|less than|fewer than|under|maximum of|max\.?)\s*$"

def getYearsRequired(description):
    text = description.lower()
    years = []

    for match in re.finditer(yearsPattern, text):
        # The text around the match, kept inside the same sentence
        before = text[max(0, match.start() - 40):match.start()]
        after = text[match.end():match.end() + 80]
        sentenceBefore = re.split(r"[.;\n]", before)[-1]
        sentenceAfter = re.split(r"[.;\n]", after)[0]

        # Skip degree substitutions and maximums
        if re.search(substituteWords, sentenceBefore + match.group(0) + sentenceAfter):
            continue
        if re.search(maximumWords, sentenceBefore):
            continue

        word = match.group(1)
        if word in numberWords:
            number = numberWords[word]
        else:
            number = int(word)

        if number <= 15:          # Threshold ignores number past this
            years.append(number)

    if len(years) == 0:
        return None
    return max(years)             # largest = strictest requirement in the post

dfJobs["years_required"] = dfJobs["description"].apply(getYearsRequired)
dfJobs.groupby("source")["years_required"].describe()

,count,mean,std,min,25%,50%,75%,max
source,,,,,,,,
hackernews,639.0,5.242567,2.811029,0.0,3.0,5.0,6.5,15.0
serpapi,627.0,3.218501,3.232489,0.0,1.0,2.0,5.0,15.0


## Full Cleaning (SerpAPI salaries from the description)

In the SerpAPI cleaning notebook, salaries only came from Google's tags. But some postings with no salary tag still list a salary in the job description, for example "Salary Range: \$114,500 - 194,700 USD."

For those postings:
- The first dollar range in the description is used as the salary.
- Hourly pay is converted to yearly pay (× 2,080 hours).
- Anything else under \$20,000 is skipped instead of guessed.

A new column, `salary_source`, records where each salary came from (Google's tag or the description). The new salaries are also checked with the same outlier limits used in the cleaning notebook.

**AI assistance:** AI was heavily involved in this step. A review by AI (Claude) first suggested that some salaries might be in the job descriptions. Claude then checked the data, found the postings with a salary in the description but no tag, helped wrote the code and the pattern matching to extract and convert them.

In [13]:
salaryPattern = r"\$\s?(\d[\d,]*(?:\.\d+)?)\s?([kK])?\s?(?:-|–|—|to)\s?\$?\s?(\d[\d,]*(?:\.\d+)?)\s?([kK])?"
hourlyWords = r"^\s*(?:usd)?\s*(?:/\s*(?:hr|hour)|per hour|an hour|hourly)"

# Turn a salary number written as text into a real number.
def toAmount(number, k):
    amount = float(number.replace(",", ""))
    if k and amount < 1000:          # "120k" -> 120,000, but "$120,000k" is not multiplied again
        amount = amount * 1000
    return amount

# Find the first salary range in a job description and return
# (yearly min, yearly max, pay unit). Returns (None, None, None)
def getSalaryFromDescription(description):
    match = re.search(salaryPattern, description)
    if match is None:
        return None, None, None

    low = toAmount(match.group(1), match.group(2) or match.group(4))   # "$90-120k": k applies to both
    high = toAmount(match.group(3), match.group(4))
    textAfter = description[match.end():match.end() + 20].lower()

    if re.search(hourlyWords, textAfter):
        return low * 2080, high * 2080, "an hour"
    if low >= 20000 and high >= low and high <= 1000000:
        return low, high, "a year"
    return None, None, None       # unclear, so it is not guessed


# Only SerpAPI rows with no salary yet (and not the 2 Canadian-dollar rows)
needsSalary = (dfJobs["source"] == "serpapi") & dfJobs["salary_min_yearly"].isnull() & (dfJobs["salary_unit"] != "non-USD")

# New column recording where each salary came from.
dfJobs["salary_source"] = None
dfJobs.loc[dfJobs["salary_min_yearly"].notna(), "salary_source"] = "tag or header"

# Go through each row that needs a salary and look for one in its description.
# If one is found, fill in the salary columns and mark the source as "description".
for i in dfJobs[needsSalary].index:
    low, high, unit = getSalaryFromDescription(dfJobs.loc[i, "description"])
    if low is not None:
        dfJobs.loc[i, "salary_min_yearly"] = low
        dfJobs.loc[i, "salary_max_yearly"] = high
        dfJobs.loc[i, "salary_unit"] = unit
        dfJobs.loc[i, "salary_source"] = "description"

# Flag outliers among the new salaries (use IQR from SerpAPI cleaning notebook)
newSalary = dfJobs["salary_source"] == "description"
tooLow = dfJobs["salary_min_yearly"] < 2500
tooHigh = dfJobs["salary_min_yearly"] > 182500
dfJobs.loc[newSalary, "is_salary_outlier"] = (tooLow | tooHigh)[newSalary]

print(f"Salaries added from descriptions: {newSalary.sum()}")
dfJobs[newSalary][["title", "salary_unit", "salary_min_yearly", "salary_max_yearly"]].head(10)

Salaries added from descriptions: 287


,title,salary_unit,salary_min_yearly,salary_max_yearly
2,Entry Level Software Engineer,a year,70828.0,120408.0
14,"Software Engineer I, AI",a year,85000.0,100000.0
19,Software Engineer - Early Career,a year,123000.0,155000.0
23,Entry Level Software Engineer,an hour,72800.0,87360.0
24,"Entry Level Software Engineer, Level 0",a year,100000.0,120000.0
25,Entry-Level Software Engineer,a year,60000.0,90000.0
33,Software Engineer (Entry level),a year,125000.0,135000.0
45,"Software Engineer, New Grad (San Mateo)",a year,110000.0,135000.0
53,Software Engineer (2027 New College Graduate),a year,72000.0,141700.0
58,Entry Level -Software Developer - Web Development,a year,170000.0,240000.0


## Degree status

The `degree_status` column shows how each posting talks about a degree:

| Label | Meaning | Example |
|---|---|---|
| **Pursuing** | For students, like internships or co-ops | "Currently pursuing a BS" |
| **Required** | The degree is a must | "Bachelor's degree required," or a degree listed under "Basic Qualifications" |
| **Preferred** | The degree is nice to have | "Master's degree preferred," or a degree listed under "Preferred Qualifications" |
| **Mentioned** | A degree is mentioned, but it's unclear if it's required | |
| **Not mentioned** | No degree is mentioned | |

If a posting mentions a degree more than once, the strongest label is used, in this order: Pursuing, Required, Preferred.

These labels are approximate. In a hand check of 20 postings, about 15 were labeled correctly. The most common mistake is a posting that lists a degree under both "Basic" and "Preferred" qualifications.

**AI assistance:** AI (Claude) wrote the rules and patterns for each label, tested them on the job descriptions, and checked the sample of 20 postings used to estimate accuracy.

In [14]:
# Is the degree required, preferred, or something the student is still working on?
# For each place a degree is mentioned, check the same sentence first, then the
# nearest heading before it (e.g. "Basic Qualifications" vs. "Preferred Qualifications").
degreeWords = (r"bachelor|master(?:'|’)?s(?:\s+degree|\s+in\b|\s+of science)|ph\.?d|doctorate|\bdegree in\b|\bb\.?s\.? in|\bm\.?s\.? in"
               r"|(?:college|university|advanced|graduate|undergraduate|related|technical|4-year|four-year|associate(?:'|’)?s?) degree")

pursuingWords = r"pursuing|currently enrolled|enrolled in|expected graduation|graduating in|graduation date|working toward"

# Words checked inside the same sentence as the degree
requiredWords = r"required|requirements?|requires|must|minimum|basic qualifications|what you(?:'|’)ll need|need to have"
preferredWords = r"preferred|prefer\b|a plus|nice to have|desired|bonus|ideally|pluses"

# Section headings checked before the degree ("a plus" on a nearby bullet is ignored here)
requiredHeadings = (r"required|requirements?|minimum|basic qualifications|what you(?:'|’)ll need|must.have"
                    r"|(?<!preferred )(?<!desired )(?<!additional )(?<!bonus )qualifications")
preferredHeadings = r"preferred (?:qualifications|skills|experience)|nice to have|bonus points|desired (?:qualifications|skills)|preferred:"

# Find where a pattern appears LAST in the text.
# Returns its position (character number), or -1 if it doesn't appear.
def lastPosition(pattern, text):
    positions = [m.start() for m in re.finditer(pattern, text)]
    if len(positions) == 0:
        return -1
    return positions[-1]

# Decide how ONE degree mention is described: Pursuing, Preferred, Required, or Mentioned.
# start/end = where the degree mention is in the text.
def getMentionStatus(text, start, end):
    # Find the sentence around the degree mention.
    # A sentence ends at a period, line break, semicolon, or bullet point.
    marks = [".", "\n", ";", "•"]
    sentenceStart = max(text.rfind(mark, 0, start) for mark in marks) + 1
    sentenceEnd = len(text)
    for mark in marks:
        position = text.find(mark, end)
        if position != -1 and position < sentenceEnd:
            sentenceEnd = position
    sentence = text[sentenceStart:sentenceEnd]

    # check to see if the sentence itself says: pursuing, preferred, or required
    if re.search(pursuingWords, sentence):
        return "Pursuing"
    if re.search(preferredWords, sentence):
        return "Preferred"
    if re.search(requiredWords, sentence):
        return "Required"

    # Nearest heading word in the 300 characters before the mention
    before = text[max(0, start - 300):start]
    requiredAt = lastPosition(requiredHeadings, before)
    preferredAt = lastPosition(preferredHeadings, before)
    if requiredAt == -1 and preferredAt == -1:
        return "Mentioned"
    if preferredAt > requiredAt:
        return "Preferred"
    return "Required"

def getDegreeStatus(description):
    text = description.lower()
    statuses = []
    for match in re.finditer(degreeWords, text):
        statuses.append(getMentionStatus(text, match.start(), match.end()))

    if len(statuses) == 0:
        return "Not mentioned"
    for status in ["Pursuing", "Required", "Preferred"]:    # strongest first
        if status in statuses:
            return status
    return "Mentioned"              # a degree is mentioned with no clear wording either way

dfJobs["degree_status"] = dfJobs["description"].apply(getDegreeStatus)
dfJobs[dfJobs["keep_for_analysis"]].groupby("source")["degree_status"].value_counts()

source      degree_status
hackernews  Not mentioned    5488
            Mentioned          95
            Required           29
            Preferred          22
serpapi     Not mentioned     466
            Required          420
            Mentioned         183
            Preferred         125
            Pursuing           38
Name: count, dtype: int64

## Requirements

Records the highest degree mentioned, whether "or equivalent experience" is accepted, security clearance, and certifications, as one string (for example `"Bachelor's, Security clearance"`). The degree part uses the same degree patterns as `degree_status`, so the two columns always agree.

True/False columns for charting:
- `mentions_degree`: any degree mention (`degree_status` is not "Not mentioned")
- `needs_degree`: a degree is clearly required (`degree_status` is "Required")
- `needs_clearance`, `needs_certification`

**AI assistance:** AI (Claude) helped write the regex patterns used to find degrees, "or equivalent experience," security clearances, and certifications in the job descriptions.

In [ ]:
def findRequirements(description):
    text = description.lower()
    found = []

    # Highest degree mentioned (only if degreeWords from the degree status step finds one)
    if re.search(degreeWords, text):
        if re.search(r"\bph\.?d\b|doctorate", text):
            found.append("PhD")
        elif re.search(r"master(?:'|’)?s(?:\s+degree|\s+in\b|\s+of science)|\bm\.?s\.? in", text):
            found.append("Master's")
        else:
            found.append("Bachelor's")

    if re.search(r"or equivalent|equivalent (?:practical |work )?experience", text):
        found.append("or equivalent experience")

    if re.search(r"clearance|ts/sci|top secret", text):
        found.append("Security clearance")

    if re.search(r"certification|certified", text):
        found.append("Certification")

    return ", ".join(found)

dfJobs["requirements"] = dfJobs["description"].apply(findRequirements)

# True/False columns for the requirements that will be charted.
# mentions_degree: any degree mention.  needs_degree: degree clearly required.
dfJobs["mentions_degree"] = dfJobs["degree_status"] != "Not mentioned"
dfJobs["needs_degree"] = dfJobs["degree_status"] == "Required"
dfJobs["needs_clearance"] = dfJobs["requirements"].str.contains("Security clearance")
dfJobs["needs_certification"] = dfJobs["requirements"].str.contains("Certification")

dfJobs[["title", "requirements", "degree_status"]].head(10)

,title,requirements,degree_status
0,Software Engineer (Entry Level),Master's,Preferred
1,"Software Engineer I, Entry-Level Graduation Da...","Master's, or equivalent experience",Preferred
2,Entry Level Software Engineer,"Master's, Security clearance",Required
3,Campus Graduate Masters Full-Time Engineer - 2...,Master's,Required
4,Entry-Level Quantitative Software Engineer I,,Not mentioned
5,Software Engineer: Credit Card/Fraud,,Not mentioned
6,software engineer (entry level),Master's,Preferred
7,Associate Software Engineer (Entry Level)/Data...,"Bachelor's, Certification",Mentioned
8,Entry Level Software Engineer Jobs,"Master's, Security clearance",Required
9,Remote Software Engineer (Java/Angular) - Entr...,,Not mentioned


## 7. Seniority and entry-level flag

Each job title is labeled with a `seniority`:

| Label | Meaning | Example |
|---|---|---|
| **Entry** | The title has an entry-level word | "Junior Developer," "Software Engineer I" |
| **Senior** | The title has a senior word only | "Senior Software Engineer," "Staff Engineer" |
| **Mixed** | The title covers more than one level, so both entry-level and senior candidates could apply | "Entry Level to Senior Level Software Developers," "Senior Associate" |
| **Unspecified** | No level is mentioned | "Software Engineer" |

**Note For Future:** A posting counts as entry-level (`is_entry_level = True`) if it is **not senior-only**, and **either** has an entry-level title **or** asks for 2 years of experience or less. Mixed postings are included.

For a strict entry-level-only analysis, use `seniority == "Entry"` instead.


**AI assistance:** A review by AI (Claude) pointed out that postings advertising several levels were being dropped as senior. AI (Claude) then created the `seniority` regex pattern and tested whether including Mixed postings changed the main results.

In [ ]:
# Entry-level words in the title
entryWords = r"junior|\bjr\b|entry|new grad|graduate|associate|early career|\bintern|apprentice|\blevel (?:1|i)\b|engineer i\b|developer i\b"
titleIsEntry = dfJobs["title"].str.lower().str.contains(entryWords, regex=True)

# Seniority: some titles advertise more than one level
# ("Entry Level to Senior Level Software Developers"), so they get their own label.
# "Engineer I" is not used here, so "Senior Software Engineer I" stays Senior.
mixedWords = r"junior|\bjr\b|entry|new grad|graduate|\bintern|associate(?! principal)"
titleHasMixedWord = dfJobs["title"].str.lower().str.contains(mixedWords, regex=True)

def getSeniority(isSenior, isEntryTitle, hasMixedWord):
    if isSenior and hasMixedWord:
        return "Mixed"
    if isSenior:
        return "Senior"
    if isEntryTitle:
        return "Entry"
    return "Unspecified"

seniorityList = []
for isSenior, isEntryTitle, hasMixedWord in zip(dfJobs["is_senior"], titleIsEntry, titleHasMixedWord):
    seniorityList.append(getSeniority(isSenior, isEntryTitle, hasMixedWord))
dfJobs["seniority"] = seniorityList

print(dfJobs[dfJobs["keep_for_analysis"]].groupby("source")["seniority"].value_counts())

# Group definition (can be changed): not senior (or mixed seniority), AND either
# the title has an entry-level word OR the posting asks for 2 years or less
notSeniorOnly = dfJobs["seniority"] != "Senior"
lowYears = dfJobs["years_required"] <= 2

dfJobs["is_entry_level"] = notSeniorOnly & (titleIsEntry | lowYears)

dfJobs[dfJobs["keep_for_analysis"]].groupby("source")["is_entry_level"].agg(["sum", "mean"])

source      seniority  
hackernews  Unspecified    3836
            Senior         1649
            Entry           111
            Mixed            38
serpapi     Entry           867
            Unspecified     237
            Senior          118
            Mixed            10
Name: count, dtype: int64


,sum,mean
source,,
hackernews,204,0.036209
serpapi,903,0.732955


## Save



In [ ]:
dfJobs.to_csv("jobs_features.csv", index=False)
print(dfJobs.shape)

(9437, 77)
